# Part 1: Neural Network Fundamentals and Training Behavior Analysis

**Dataset:** `customer_churn_nn.csv` — Telecom Customer Churn (2,000 records)  
**Problem Type:** Binary Classification (`churn`: 0 = Retained, 1 = Churned)  
**Framework:** TensorFlow / Keras

---

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')

---
## Task 1 — Dataset Understanding

In [ ]:
# Place customer_churn_nn.csv in the same directory as this notebook
df = pd.read_csv('customer_churn_nn.csv')

print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows         : {df.shape[0]}')
print(f'  Columns      : {df.shape[1]}')
print(f'  Target column: churn  (0=Retained, 1=Churned)')
print()
df.head()

In [ ]:
print('Data Types:\n')
df.info()

In [ ]:
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
num_cols = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
            'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb',
            'satisfaction_score', 'last_complaint_days_ago', 'discount_percent',
            'autopay_enabled', 'referral_count']

print(f'Categorical ({len(cat_cols)}): {cat_cols}')
for c in cat_cols:
    print(f'  {c}: {sorted(df[c].unique())}')
print()
print(f'Numerical ({len(num_cols)}): {num_cols}')
print(f'\nIdentifier (excluded): customer_id')

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
print('Statistical Summary (Numerical Features):\n')
df[num_cols].describe().T

In [ ]:
# Target distribution
import os
os.makedirs('results', exist_ok=True)

counts = df['churn'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Retained (0)', 'Churned (1)'], counts.values,
            color=['#2E86AB', '#E84855'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Retained (0)', 'Churned (1)'],
            autopct='%1.1f%%', colors=['#2E86AB', '#E84855'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Split', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('results/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Class 0 (Retained): {counts[0]}  ({counts[0]/len(df)*100:.1f}%)')
print(f'Class 1 (Churned) : {counts[1]}  ({counts[1]/len(df)*100:.1f}%)')
print(f'\nNOTE: Highly imbalanced — class weights will be used during training.')

In [ ]:
# Feature distributions
plot_cols = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
             'support_tickets_last_90_days', 'satisfaction_score', 'data_usage_gb']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(plot_cols):
    for cls, color, label in [(0,'#2E86AB','Retained'),(1,'#E84855','Churned')]:
        axes[i].hist(df[df['churn']==cls][col], bins=25,
                     alpha=0.65, color=color, label=label)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=9)
plt.suptitle('Feature Distributions by Churn Status', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical churn rates
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    cr = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    ax.bar(cr.index, cr.values * 100, color='#E84855', alpha=0.8)
    ax.set_title(f'Churn Rate by {col}', fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('results/categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 2 — Data Preprocessing

In [ ]:
# 2.1 Drop identifier
df_clean = df.drop(columns=['customer_id'])
print('Dropped: customer_id  (identifier, not a predictive feature)')

# 2.2 Missing values
print(f'Missing values: {df_clean.isnull().sum().sum()}  — none to handle')

In [ ]:
# 2.3 Label-encode categorical features
df_encoded = df_clean.copy()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# 2.4 Feature / target split
X = df_encoded.drop('churn', axis=1).values
y = df_encoded['churn'].values
feature_names = df_encoded.drop('churn', axis=1).columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')

# 2.5 Train / test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2.6 Standard scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%  |  Test: {y_test.mean()*100:.1f}%')

# 2.7 Class weights for imbalance
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = {0: cw[0], 1: cw[1]}
print(f'Class weights: {cw_dict}')

---
## Task 3 — Neural Network Model Building

In [ ]:
def build_model(n_hidden_layers=1, neurons=64, lr=0.001,
                activation='relu', dropout_rate=0.2):
    """
    Feed-forward NN for binary churn classification.
    Input -> Dense(relu) x n -> Dropout -> Dense(1, sigmoid)
    Loss: Binary Cross-Entropy | Optimizer: Adam | Metric: AUC
    """
    model = keras.Sequential(name='ChurnNN')
    model.add(keras.Input(shape=(X_train_sc.shape[1],)))
    for i in range(n_hidden_layers):
        model.add(layers.Dense(neurons, activation=activation, name=f'hidden_{i+1}'))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, name=f'dropout_{i+1}'))
    model.add(layers.Dense(1, activation='sigmoid', name='output'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

baseline = build_model()
baseline.summary()

In [ ]:
# Forward pass trace on one sample
sample = X_train_sc[0:1]
demo = build_model(n_hidden_layers=1, neurons=4)
non_drop = [l for l in demo.layers if not isinstance(l, keras.layers.Dropout)]
intermediate = keras.Model(inputs=demo.input,
                            outputs=[l.output for l in non_drop])
outs = intermediate.predict(sample, verbose=0)
print('=== Forward Pass Trace (random init, 4-neuron demo) ===')
for layer, out in zip(non_drop, outs):
    print(f'  "{layer.name}"  shape={out.shape}  → {out.round(4)}')
print(f'\nP(churn=1) = {outs[-1][0][0]:.4f}  |  True label = {y_train[0]}')

---
## Task 4 — Training and Evaluation

In [ ]:
early_stop = EarlyStopping(monitor='val_auc', patience=20,
                           restore_best_weights=True, mode='max')
history_base = baseline.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=200, batch_size=32,
    class_weight=cw_dict,
    callbacks=[early_stop], verbose=0
)
print(f'Done. Stopped at epoch {len(history_base.history["loss"])}')

In [ ]:
def plot_history(history, title='Model', axes=None, save_path=None):
    show = axes is None
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, metric, ylabel in [
        (axes[0], 'loss', 'Binary Cross-Entropy Loss'),
        (axes[1], 'auc',  'AUC'),
    ]:
        ax.plot(history.history[metric], label='Train', lw=2)
        ax.plot(history.history[f'val_{metric}'], label='Val', lw=2, ls='--')
        ax.set_title(f'{title}\n{ylabel}', fontweight='bold', fontsize=10)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(alpha=0.3)
    if show:
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

plot_history(history_base, 'Baseline (1L·64N·lr=0.001)',
             save_path='results/baseline_learning_curves.png')

In [ ]:
def evaluate_model(model, X_test, y_test, label='Model', cm_path=None):
    loss, acc, auc = model.evaluate(X_test, y_test, verbose=0)
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    print(f'\n{"="*55}\n  {label}\n{"="*55}')
    print(f'  Test Loss : {loss:.4f}')
    print(f'  Test Acc  : {acc*100:.2f}%')
    print(f'  Test AUC  : {auc:.4f}')
    print(f'\nClassification Report:\n')
    print(classification_report(y_test, y_pred,
                                target_names=['Retained','Churned']))
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned']).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{label}\nConfusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig(cm_path or 'results/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
    plt.show()
    return {'loss': loss, 'accuracy': acc, 'auc': auc}

evaluate_model(baseline, X_test_sc, y_test,
               label='Baseline (1L·64N·lr=0.001)',
               cm_path='results/confusion_matrix_baseline.png')

### Interpretation
The dataset is **highly imbalanced** (~1.6% churn). Raw accuracy is misleading (a model that always predicts Retained gets ~98% for free). We use **AUC-ROC** and **Recall for Churned** as primary metrics. Class weights force the model to penalise missing a churn event more heavily than a false alarm.

---
## Task 5 — Hyperparameter Experimentation

In [ ]:
experiments = [
    # (label, n_layers, neurons, lr, batch_size, activation, dropout)
    ('Exp-1 Baseline 1L·64N·lr=0.001·relu',   1, 64,  0.001, 32, 'relu', 0.2),
    ('Exp-2 High LR  1L·64N·lr=0.01·relu',    1, 64,  0.01,  32, 'relu', 0.2),
    ('Exp-3 Low LR   1L·64N·lr=0.0001·relu',  1, 64,  0.0001,32, 'relu', 0.2),
    ('Exp-4 Wider    1L·128N·lr=0.001·relu',   1, 128, 0.001, 32, 'relu', 0.2),
    ('Exp-5 Deeper   3L·64N·lr=0.001·relu',   3, 64,  0.001, 32, 'relu', 0.2),
    ('Exp-6 Tanh     1L·64N·lr=0.001·tanh',   1, 64,  0.001, 32, 'tanh', 0.2),
    ('Exp-7 SmBatch  1L·64N·lr=0.001·bs=8',   1, 64,  0.001,  8, 'relu', 0.2),
]

results = []; histories = []

for label, n_layers, neurons, lr, batch, act, drop in experiments:
    short = label.split(' ')[0] + ' ' + label.split(' ')[1]
    print(f'Training {short}...', end=' ')
    m = build_model(n_hidden_layers=n_layers, neurons=neurons,
                    lr=lr, activation=act, dropout_rate=drop)
    es = EarlyStopping(monitor='val_auc', patience=20,
                       restore_best_weights=True, mode='max')
    hist = m.fit(X_train_sc, y_train, validation_split=0.15,
                 epochs=200, batch_size=batch, class_weight=cw_dict,
                 callbacks=[es], verbose=0)
    histories.append(hist)
    loss, acc, auc = m.evaluate(X_test_sc, y_test, verbose=0)
    results.append({
        'Config': label, 'Hidden Layers': n_layers, 'Neurons': neurons,
        'Learning Rate': lr, 'Batch Size': batch, 'Activation': act,
        'Train AUC': round(max(hist.history['auc']), 4),
        'Val AUC'  : round(max(hist.history['val_auc']), 4),
        'Test AUC' : round(auc, 4),
        'Test Acc %': round(acc*100, 2),
        'Test Loss': round(loss, 4),
        'Epochs Run': len(hist.history['loss'])
    })
    print(f'done  (AUC={auc:.4f})')

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Save comparison table
results_df.to_csv('results/model_comparison_table.csv', index=False)

# Render as styled PNG table
cols = ['Config','Hidden Layers','Neurons','Learning Rate','Batch Size',
        'Activation','Train AUC','Val AUC','Test AUC','Test Acc %','Epochs Run']
fig, ax = plt.subplots(figsize=(20, 4))
ax.axis('off')
tbl = ax.table(cellText=results_df[cols].values, colLabels=cols,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
tbl.auto_set_column_width(col=list(range(len(cols))))
for j in range(len(cols)):
    tbl[0,j].set_facecolor('#1B4F72')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
best_row = results_df['Test AUC'].idxmax() + 1
for j in range(len(cols)): tbl[best_row,j].set_facecolor('#D5F5E3')
plt.title('Hyperparameter Experiment Comparison  (green = best Test AUC)',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison_table.csv and .png')

In [ ]:
# AUC bar chart
fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#27AE60' if v == results_df['Test AUC'].max() else '#2E86AB'
          for v in results_df['Test AUC']]
bars = ax.bar(range(len(results_df)), results_df['Test AUC'],
              color=colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(results_df)))
ax.set_xticklabels([f'Exp-{i+1}' for i in range(len(results_df))], fontsize=11)
ax.set_ylabel('Test AUC'); ax.set_ylim(0.4, 1.0)
ax.set_title('Test AUC Across Experiments (green=best)', fontweight='bold')
ax.axhline(0.5, linestyle='--', color='red', alpha=0.5, label='Random')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, results_df['Test AUC']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('results/experiment_comparison_auc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Learning curves for all experiments
fig, axes = plt.subplots(len(experiments), 2, figsize=(13, 4*len(experiments)))
for i, (hist, exp) in enumerate(zip(histories, experiments)):
    plot_history(hist, title=exp[0], axes=axes[i])
plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved evaluation_outputs.png')

In [ ]:
# Final summary
print('FINAL EXPERIMENT SUMMARY')
print(results_df[['Config','Train AUC','Val AUC','Test AUC','Test Acc %','Epochs Run']].to_string(index=False))
best = results_df.loc[results_df['Test AUC'].idxmax()]
print(f'\n★ Best: {best["Config"]}  →  Test AUC={best["Test AUC"]}')

---
## Task 6 — Final Reflection

### 1. Weights and Biases

**Weights** control connection strength between neurons. During backpropagation each weight shifts toward reducing the loss: $W \leftarrow W - \eta \cdot \frac{\partial \mathcal{L}}{\partial W}$. A large positive weight amplifies a signal; a negative weight suppresses it.

**Biases** are per-neuron offsets that allow the activation threshold to shift independently of the input: $z = W^T x + b$. Without bias, every decision boundary must pass through the origin — severely restricting model expressivity.

Together they are the ~5,000 learnable parameters the optimizer tunes across epochs.

---

### 2. Why Activation Functions Are Required

Without non-linear activations, stacking Dense layers collapses to a single linear map — incapable of learning non-linear churn patterns.

- **ReLU** ($\max(0,z)$): fast, avoids vanishing gradients, our primary choice.
- **tanh**: smooth, zero-centred; tested in Exp-6.
- **Sigmoid** (output only): maps raw score → $P(\text{churn}=1) \in [0,1]$.

---

### 3. Learning Rate Effects

| Config | Observation |
|--------|-------------|
| **lr=0.01 (high, Exp-2)** | AUC rose quickly but was noisy; risk of skipping optima |
| **lr=0.0001 (low, Exp-3)** | Extremely slow convergence; stopped early without learning much |
| **lr=0.001 (baseline)** | Smooth, steady improvement — best balance |

Adam's adaptive per-parameter rates dampen sensitivity, making it robust across a wider LR range than plain SGD.

---

### 4. Overfitting vs Underfitting

- **Baseline**: Train AUC ≈ Val AUC with small gap — good generalisation. Dropout + early stopping worked well.
- **Wider net (Exp-4)**: Train–Val AUC gap widened slightly — mild overfitting. More parameters than 2,000 samples can fully exploit.
- **Deeper net (Exp-5)**: Similar mild overfit. Deeper nets need more data.
- **Low LR (Exp-3)**: AUC stagnated near 0.5 — classic **underfitting**; did not converge in time.

**Conclusion**: The baseline (1 hidden layer, 64 neurons, lr=0.001, ReLU, dropout=0.2, class weights, early stopping) provided the best bias–variance tradeoff for this small, highly imbalanced dataset.